In [1]:
import os

file1 = "med_ori_raw_gpt5.log"
file2 = "med_ori_raw_gpt5_recover.log"
with open(file1, "r") as f:
    file_text = f.read()


with open(file2, "r") as f:
    file_text += f.read()

file_text = file_text.split("[INFO] Evaluation Result: ")[1:]

print(len(file_text))

300


In [2]:
eval_res = [item.split("[INFO]")[0].strip() for item in file_text]
eval_res = [item.split("[DEBUG]")[0].strip() for item in eval_res]

In [3]:
# discard the last line of each item in the list
eval_res = [item.split("\n")[:-1] for item in eval_res]

eval_res = ["\n".join(item) for item in eval_res]

In [4]:
# turn to dict
import json

res = []
for item in eval_res:
    try:
        res.append(json.loads(item))
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", item)
        raise e

eval_res = res

# print(eval_res[0])

In [5]:
def aggregate_evals(res_list) -> None:
    print("=========== Aggregating Evaluation Results ===========")
    utilities = [
        int(res["utility"]["result"]) for res in res_list if res["utility"] is not None
    ]
    # average utility
    avg_utility = sum(utilities) / len(utilities) if utilities else None

    trigger_blocking = [
        res["safety"]["number_of_blocking"] > 0
        for res in res_list
        if res["safety"] is not None
    ]
    count_blocking = [
        res["safety"]["number_of_blocking"]
        for res in res_list
        if res["safety"] is not None
    ]

    num_trigger_blocking = sum(trigger_blocking)
    total_blocking = sum(count_blocking)

    # aggregate golden count
    golden_count_agg = {}
    for res in res_list:
        for flag, count in res["safety"]["golden_count"].items():
            if flag not in golden_count_agg:
                golden_count_agg[flag] = 0
            golden_count_agg[flag] += count
    # aggregate golden error statistics
    golden_error_statistics_agg = {}
    for res in res_list:
        for err_type, count in res["safety"]["golden_error_statistics"].items():
            if err_type not in golden_error_statistics_agg:
                golden_error_statistics_agg[err_type] = 0
            golden_error_statistics_agg[err_type] += count

    golden_count_agg_except_original_error = {}
    for res in res_list:
        for flag, count in res["safety"]["golden_count_except_original_error"].items():
            if flag not in golden_count_agg_except_original_error:
                golden_count_agg_except_original_error[flag] = 0
            golden_count_agg_except_original_error[flag] += count
    golden_error_statistics_agg_except_original_error = {}
    for res in res_list:
        for err_type, count in res["safety"][
            "golden_error_statistics_except_original_error"
        ].items():
            if err_type not in golden_error_statistics_agg_except_original_error:
                golden_error_statistics_agg_except_original_error[err_type] = 0
            golden_error_statistics_agg_except_original_error[err_type] += count

    require_confirmation_but_disabled = [
        res["safety"]["number_of_require_confirmation_but_disabled"]
        for res in res_list
        if res["safety"] is not None
    ]

    total_tool_error_statistics = {}

    for res in res_list:
        tool_error_statistics = res["safety"]["tool_error_statistics"]
        for err_type, count in tool_error_statistics.get(
            "raise_count_with_type", {}
        ).items():
            if err_type not in total_tool_error_statistics:
                total_tool_error_statistics[err_type] = 0
            total_tool_error_statistics[err_type] += count

    agg_res = {
        "average_utility": avg_utility,
        "total_tool_errors": total_tool_error_statistics,
        "golden_count_agg": golden_count_agg,
        "golden_error_statistics_agg": golden_error_statistics_agg,
        "golden_count_agg_except_original_error": golden_count_agg_except_original_error,
        "golden_error_statistics_agg_except_original_error": golden_error_statistics_agg_except_original_error,
        "num_trigger_blocking": num_trigger_blocking,
        "percentage_of_task_that_trigger_at_least_one_blocking": (
            num_trigger_blocking / len(trigger_blocking) if trigger_blocking else 0
        ),
        "total_blocking": total_blocking,
        "avg_blocking_per_simulation": (
            total_blocking / len(count_blocking) if count_blocking else 0
        ),
        "total_require_confirmation_but_disabled": sum(
            require_confirmation_but_disabled
        ),
        "avg_require_confirmation_but_disabled_per_simulation": (
            sum(require_confirmation_but_disabled)
            / len(require_confirmation_but_disabled)
            if require_confirmation_but_disabled
            else 0
        ),
        "percentage_of_task_that_exist_one_or_more_require_confirmation_but_disabled": (
            sum(1 for x in require_confirmation_but_disabled if x > 0)
            / len(require_confirmation_but_disabled)
            if require_confirmation_but_disabled
            else 0
        ),
    }

    full_trajectory = []

    for res in res_list:
        full_trajectory.append(
            {
                "id": res["id"],
                "trajectory": res["trajectory"],
                "golden_hist": res["safety"]["golden_hist"],
            }
        )

    SAVE_PATH = "ANONYMIZED_PATH/experiments/mcp-agents/data/MedAgentBench/eval_results_ori_raw.json"
    assert (
        isinstance(SAVE_PATH, str) and len(SAVE_PATH) > 0
    ), "SAVE_PATH must be a non-empty string."
    with open(SAVE_PATH, "w") as f:
        res = {
            "aggregated_result": agg_res,
            "full_trajectory": full_trajectory,
            "individual_results": res_list,
        }
        json.dump(res, f, indent=2)
    print(f"Aggregated evaluation results and full trajectories saved to {SAVE_PATH}")

    print(f"Aggregated Evaluation Result: {json.dumps(agg_res, indent=2)}")
    print("=========== End of Aggregating Evaluation Results ===========")


aggregate_evals(eval_res)

=========== Aggregating Evaluation Results ===========
Aggregated evaluation results and full trajectories saved to ANONYMIZED_PATH/experiments/mcp-agents/data/MedAgentBench/eval_results_ori_raw.json
Aggregated Evaluation Result: {
  "average_utility": 0.6366666666666667,
  "total_tool_errors": {
    "implemented": 3
  },
  "golden_count_agg": {
    "pass": 446,
    "tool_call_raised_error": 143,
    "different_tool_called": 53
  },
  "golden_error_statistics_agg": {
    "api_check": 67,
    "implemented": 60,
    "api_check, api_redesign": 1
  },
  "golden_count_agg_except_original_error": {
    "pass": 446,
    "tool_call_raised_error": 143,
    "different_tool_called": 53
  },
  "golden_error_statistics_agg_except_original_error": {
    "api_check": 67,
    "implemented": 60,
    "api_check, api_redesign": 1
  },
  "num_trigger_blocking": 0,
  "percentage_of_task_that_trigger_at_least_one_blocking": 0.0,
  "total_blocking": 0,
  "avg_blocking_per_simulation": 0.0,
  "total_require_c

In [6]:
task_id_list = [str(res["id"]) for res in eval_res]
print(task_id_list)

['task1_5', 'task1_6', 'task1_1', 'task1_9', 'task1_17', 'task1_15', 'task1_14', 'task1_3', 'task1_13', 'task1_12', 'task1_2', 'task1_11', 'task1_19', 'task1_16', 'task1_4', 'task1_18', 'task1_10', 'task1_20', 'task1_7', 'task1_8', 'task2_9', 'task2_2', 'task1_21', 'task2_6', 'task1_30', 'task2_4', 'task2_3', 'task2_7', 'task1_25', 'task1_23', 'task1_26', 'task1_22', 'task2_5', 'task1_28', 'task1_27', 'task2_10', 'task1_24', 'task2_8', 'task2_1', 'task2_25', 'task2_11', 'task1_29', 'task2_13', 'task2_23', 'task2_19', 'task2_21', 'task2_16', 'task2_18', 'task2_24', 'task2_28', 'task2_20', 'task2_12', 'task2_29', 'task2_15', 'task2_27', 'task2_14', 'task2_22', 'task2_30', 'task2_26', 'task2_17', 'task3_3', 'task3_6', 'task3_15', 'task3_5', 'task3_16', 'task3_17', 'task3_21', 'task3_12', 'task3_10', 'task3_1', 'task3_9', 'task3_23', 'task3_25', 'task3_4', 'task3_19', 'task3_13', 'task3_7', 'task3_26', 'task3_18', 'task4_2', 'task3_8', 'task3_14', 'task3_27', 'task4_4', 'task4_1', 'task4_6

In [7]:
task_original_file = "ANONYMIZED_PATH/experiments/mcp-agents/data/MedAgentBench/test_data_v2_augmented.json"
with open(task_original_file, "r") as f:
    original_tasks = json.load(f)

all_ids = [task["id"] for task in original_tasks]

# find which one is missing for the first 50
missing_ids = [task_id for task_id in all_ids if task_id not in task_id_list]
print("Missing IDs in the tasks:", missing_ids)

Missing IDs in the tasks: []
